# Original Data - Find hyperparameter using Optuna

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import warnings
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings(
    "ignore",
    message="use_inf_as_na option is deprecated",
    category=FutureWarning)

warnings.filterwarnings(
    "ignore",
    message="The default of observed=False is deprecated",
    category=FutureWarning)

warnings.filterwarnings(
    "ignore",
    message="When grouping with a length-1 list-like",
    category=FutureWarning)

warnings.simplefilter(action='ignore', category=FutureWarning)

train = pd.read_csv('../Data/train.csv', index_col = 'id')
test = pd.read_csv('../Data/test.csv', index_col = 'id')
target = 'diagnosed_diabetes'

def overview(df, nunique_threshold=15):
    """
    Prints an overview of the DataFrame.
    - Ignores the last column (assumed target).
    - Counts numerical and categorical features.
    - Converts low-cardinality numeric features to categorical.
    """

    import pandas as pd

    # Drop last column (assumed target)
    features = df.iloc[:, :-1]

    # Initial type detection
    num_cols = features.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = features.select_dtypes(include=['object', 'category']).columns.tolist()

    # Convert low-cardinality numeric columns to categorical
    converted_cols = []
    for col in num_cols.copy():
        if features[col].nunique() <= nunique_threshold:
            df[col] = df[col].astype('category')
            num_cols.remove(col)
            cat_cols.append(col)
            converted_cols.append(col)

    print("===== DATA OVERVIEW =====")
    print(f"Total features (excluding last column): {features.shape[1]}")
    print(f"Numeric features: {len(num_cols)}")
    print(f"Categorical features: {len(cat_cols)}")

    if converted_cols:
        print(f"\nConverted to categorical (nunique ≤ {nunique_threshold}):")
        for col in converted_cols:
            print(f"  - {col}")

    print("\n--- Categorical Columns Detail ---")
    for col in cat_cols:
        print(f"{col}: {df[col].nunique()} unique values")

    return num_cols, cat_cols
    
num_cols, cat_cols = overview(train)

for col in train.select_dtypes(include='object').columns:
    train[col] = pd.Categorical(train[col])

for col in train.columns:
    if col in test.columns:
        test[col] = test[col].astype(train[col].dtype)

===== DATA OVERVIEW =====
Total features (excluding last column): 24
Numeric features: 14
Categorical features: 10

Converted to categorical (nunique ≤ 15):
  - alcohol_consumption_per_week
  - family_history_diabetes
  - hypertension_history
  - cardiovascular_history

--- Categorical Columns Detail ---
gender: 3 unique values
ethnicity: 5 unique values
education_level: 4 unique values
income_level: 5 unique values
smoking_status: 3 unique values
employment_status: 4 unique values
alcohol_consumption_per_week: 9 unique values
family_history_diabetes: 2 unique values
hypertension_history: 2 unique values
cardiovascular_history: 2 unique values


In [2]:
neg = np.sum(train[target] == 0)
pos = np.sum(train[target] == 1)
scale_pos_weight = neg / pos
scale_pos_weight
X = train.drop('diagnosed_diabetes',axis=1)
y = train['diagnosed_diabetes']

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_curve, auc, roc_auc_score

X_train, X_val, y_train, y_val = train_test_split(X,y, stratify = y, random_state=42)

## Optuna XGBoost:

In [9]:
optuna.logging.set_verbosity(optuna.logging.INFO)
import xgboost as xgb
import optuna
from sklearn.metrics import roc_auc_score

# ----------------------------
# Prepare DMatrix
# ----------------------------
dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

def objective(trial):

    booster = trial.suggest_categorical("booster", ["gbtree", "dart"])

    param = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "booster": booster,
        "seed": 42,

        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "eta": trial.suggest_float("eta", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        "gamma": trial.suggest_float("gamma", 0, 5),
        "alpha": trial.suggest_float("alpha", 0, 5),
        "lambda": trial.suggest_float("lambda", 0, 5),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),

        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 10),
        "scale_pos_weight": scale_pos_weight,
        "n_jobs": -1,
    }

    if booster == "dart":
        param.update({
            "rate_drop": trial.suggest_float("rate_drop", 0.0, 0.5),
            "skip_drop": trial.suggest_float("skip_drop", 0.0, 0.5),
            "sample_type": trial.suggest_categorical(
                "sample_type", ["uniform", "weighted"]
            ),
            "normalize_type": trial.suggest_categorical(
                "normalize_type", ["tree", "forest"]
            ),
        })

    num_boost_round = trial.suggest_int("num_boost_round", 200, 1200)

    bst = xgb.train(
        params=param,
        dtrain=dtrain,
        num_boost_round=num_boost_round,
        evals=[(dval, "validation")],
        early_stopping_rounds=20,
        verbose_eval=False,
    )

    y_pred = bst.predict(
        dval, iteration_range=(0, bst.best_iteration + 1)
    )

    return roc_auc_score(y_val, y_pred)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# ----------------------------
# Results
# ----------------------------
print("Best ROC-AUC:", study.best_value)
print("Best params:", study.best_params)


[I 2026-01-03 00:30:47,004] A new study created in memory with name: no-name-aa148aac-d973-4e96-800b-c8df097dafec
[I 2026-01-03 00:31:02,261] Trial 0 finished with value: 0.712773167777401 and parameters: {'booster': 'gbtree', 'max_depth': 6, 'eta': 0.02266273864076758, 'subsample': 0.7443542596701944, 'colsample_bytree': 0.966528853202976, 'gamma': 3.1153918344272795, 'alpha': 3.5478267659912293, 'lambda': 2.5739816335081436, 'min_child_weight': 7, 'max_cat_to_onehot': 7, 'num_boost_round': 243}. Best is trial 0 with value: 0.712773167777401.
[I 2026-01-03 01:26:13,557] Trial 1 finished with value: 0.7205319694724729 and parameters: {'booster': 'dart', 'max_depth': 8, 'eta': 0.052835387119879194, 'subsample': 0.7685765386006065, 'colsample_bytree': 0.7767727694848855, 'gamma': 1.1617426594298574, 'alpha': 2.7267748342594227, 'lambda': 2.010964916649549, 'min_child_weight': 9, 'max_cat_to_onehot': 3, 'rate_drop': 0.19163422548357556, 'skip_drop': 0.17663858684805206, 'sample_type': 'un

Best ROC-AUC: 0.7261150854440153
Best params: {'booster': 'gbtree', 'max_depth': 5, 'eta': 0.07815109367099736, 'subsample': 0.8712506708090506, 'colsample_bytree': 0.7681646250943356, 'gamma': 0.8918764623607585, 'alpha': 4.470940159950184, 'lambda': 4.022292589621206, 'min_child_weight': 2, 'max_cat_to_onehot': 7, 'num_boost_round': 797}


Best ROC-AUC: 0.7261150854440153
Best params: {'booster': 'gbtree', 'max_depth': 5, 'eta': 0.07815109367099736, 'subsample': 0.8712506708090506, 'colsample_bytree': 0.7681646250943356, 'gamma': 0.8918764623607585, 'alpha': 4.470940159950184, 'lambda': 4.022292589621206, 'min_child_weight': 2, 'max_cat_to_onehot': 7, 'num_boost_round': 797}


## Optuna LightGBM:

In [4]:
from lightgbm import LGBMClassifier, early_stopping
from sklearn.metrics import roc_auc_score


# ----------------------------
# Define objective for Optuna
# ----------------------------
def objective(trial):
    model = LGBMClassifier(
        objective='binary',
        boosting_type="gbdt",
        num_leaves=trial.suggest_int('num_leaves', 16, 256),
        max_depth=trial.suggest_int('max_depth', 3, 12),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 100),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha=trial.suggest_float('reg_alpha', 0.0, 5.0),
        reg_lambda=trial.suggest_float('reg_lambda', 0.0, 5.0),
        scale_pos_weight=4.5,
        n_estimators=trial.suggest_int('num_boost_round', 100, 2000),
        n_jobs=-1, verbosity=-1 
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        categorical_feature=cat_cols,
        callbacks=[early_stopping(stopping_rounds=50, verbose=False)]
    )

    y_pred = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred)
    return auc

# ----------------------------
# Run Optuna study
# ----------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# ----------------------------
# Results
# ----------------------------
print("Best ROC-AUC:", study.best_value)
print("Best params:", study.best_params)

[I 2026-01-02 14:40:32,448] A new study created in memory with name: no-name-d9821685-38da-4c44-8b59-b95d8519ca82
[I 2026-01-02 14:40:34,431] Trial 0 finished with value: 0.6846734497805264 and parameters: {'num_leaves': 48, 'max_depth': 6, 'learning_rate': 0.17117099998960444, 'min_child_samples': 16, 'subsample': 0.8859402230647664, 'colsample_bytree': 0.9704059203656796, 'reg_alpha': 0.5233288304326922, 'reg_lambda': 2.998846820560312, 'num_boost_round': 940}. Best is trial 0 with value: 0.6846734497805264.
[I 2026-01-02 14:40:36,199] Trial 1 finished with value: 0.6949668162478246 and parameters: {'num_leaves': 54, 'max_depth': 7, 'learning_rate': 0.010543198641219645, 'min_child_samples': 19, 'subsample': 0.7201054708006422, 'colsample_bytree': 0.7370713037330779, 'reg_alpha': 2.5219854505205936, 'reg_lambda': 2.3268989032156364, 'num_boost_round': 842}. Best is trial 1 with value: 0.6949668162478246.
[I 2026-01-02 14:40:38,126] Trial 2 finished with value: 0.6856254424399733 and 

Best ROC-AUC: 0.7055465485554286
Best params: {'num_leaves': 227, 'max_depth': 11, 'learning_rate': 0.015276056330994207, 'min_child_samples': 61, 'subsample': 0.6696617268568047, 'colsample_bytree': 0.6678441905031072, 'reg_alpha': 0.5113089859188895, 'reg_lambda': 1.515216046045771, 'num_boost_round': 353}


Best ROC-AUC: 0.7055465485554286
Best params: {'num_leaves': 227, 'max_depth': 11, 'learning_rate': 0.015276056330994207, 'min_child_samples': 61, 'subsample': 0.6696617268568047, 'colsample_bytree': 0.6678441905031072, 'reg_alpha': 0.5113089859188895, 'reg_lambda': 1.515216046045771, 'num_boost_round': 353}

## Optuna Catboost:

In [6]:
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 2000),  # num_boost_round
        'depth': trial.suggest_int('depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'scale_pos_weight': scale_pos_weight,
        'thread_count': -1,
        'task_type': 'CPU',
        'random_seed': 42,
        'logging_level': 'Silent'
    }

    model = CatBoostClassifier(**params)
    
    train_pool = Pool(X_train, y_train, cat_features=cat_cols)
    val_pool = Pool(X_val, y_val, cat_features=cat_cols)
    
    model.fit(
        train_pool,
        eval_set=val_pool,
        early_stopping_rounds=10,
        verbose=False
    )
    
    y_pred = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred)
    return auc  # Optuna maximizes by default

# ----------------------------
# Run Optuna study
# ----------------------------
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

# ----------------------------
# Best results
# ----------------------------
print("Best ROC-AUC:", study.best_value)
print("Best params:", study.best_params)

[I 2026-01-02 15:04:54,853] A new study created in memory with name: no-name-a4ec87fd-4851-4a8f-9099-f5e443445e37


[I 2026-01-02 15:09:17,230] Trial 0 finished with value: 0.7135262827383204 and parameters: {'iterations': 603, 'depth': 8, 'learning_rate': 0.01766425195580666, 'l2_leaf_reg': 2.5713205506406314, 'bagging_temperature': 0.21049928618637104, 'border_count': 223}. Best is trial 0 with value: 0.7135262827383204.
[I 2026-01-02 15:10:13,135] Trial 1 finished with value: 0.7220083315781558 and parameters: {'iterations': 717, 'depth': 9, 'learning_rate': 0.2326909780413103, 'l2_leaf_reg': 5.576771800302913, 'bagging_temperature': 0.456532210111364, 'border_count': 189}. Best is trial 1 with value: 0.7220083315781558.
[I 2026-01-02 15:15:47,454] Trial 2 finished with value: 0.7230800449222023 and parameters: {'iterations': 1231, 'depth': 5, 'learning_rate': 0.03836467540612431, 'l2_leaf_reg': 6.6791027217242025, 'bagging_temperature': 0.6108620824185746, 'border_count': 205}. Best is trial 2 with value: 0.7230800449222023.
[I 2026-01-02 15:17:20,969] Trial 3 finished with value: 0.715362225008

Best ROC-AUC: 0.7253095721926582
Best params: {'iterations': 1162, 'depth': 7, 'learning_rate': 0.07750133623087592, 'l2_leaf_reg': 6.040419312733866, 'bagging_temperature': 0.5698939842266584, 'border_count': 177}


Best ROC-AUC: 0.7253095721926582
Best params: {'iterations': 1162, 'depth': 7, 'learning_rate': 0.07750133623087592, 'l2_leaf_reg': 6.040419312733866, 'bagging_temperature': 0.5698939842266584, 'border_count': 177}